# 04 - MLOps avec MLflow : tracking, artefacts, registry et validation gate

Objectif : repondre a la question du CDO : **Comment deployer une nouvelle version sans casser la production ?**

Ce notebook met en place une boucle MLOps minimale :

- creation de l'experience MLflow `fraud-detection-paytrack` ;
- entrainement de plusieurs strategies XGBoost ;
- logging des parametres, metriques et artefacts ;
- logging des courbes Precision-Recall ;
- logging des matrices de confusion ;
- selection du meilleur modele ;
- simulation d'une validation gate ;
- enregistrement du meilleur modele dans le Model Registry local.


## 0. Installation et imports

Si besoin :

```bash
pip install mlflow polars pyarrow pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn
```

Option UI MLflow locale :

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 8080
```


In [1]:
from pathlib import Path
import gc
import json
import pickle
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.tracking import MlflowClient

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBClassifier
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
ASSETS_DIR = PROJECT_ROOT / "docs" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)
MLFLOW_DB_PATH = PROJECT_ROOT / "mlflow.db"
MLARTIFACTS_DIR = PROJECT_ROOT / "mlartifacts"
MLARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
TRAIN_IDENTITY_PATH = DATA_DIR / "train_identity.csv"

for path in [TRAIN_TRANSACTION_PATH, TRAIN_IDENTITY_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path.resolve()}")


/home/fenitra/.pyenv/versions/fraude/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration MLflow et gate

La validation gate simule une regle de promotion : un modele est enregistrable seulement si ses metriques depassent des seuils minimums.

Les seuils ci-dessous sont adaptes au POC actuel. Dans une vraie production, ils seraient fixes par le CDO et le responsable risque.


In [2]:
EXPERIMENT_NAME = "fraud-detection-paytrack"
REGISTERED_MODEL_NAME = "paytrack-fraud-xgboost"
VALID_SIZE = 0.20
FEATURE_SET = "compact"
MAX_TRAIN_ROWS_FOR_RESAMPLING = 150_000
DECISION_THRESHOLD = 0.25

# Gate volontairement realiste pour l'etat actuel du POC.
GATE_MIN_AUPRC = 0.45
GATE_MIN_RECALL = 0.40
GATE_MAX_ALERT_RATE = 0.05
GATE_MAX_INFERENCE_MS_PER_ROW = 50.0

BASE_XGB_PARAMS = dict(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# MLflow 3 bloque le backend filesystem par defaut. On utilise donc SQLite pour le tracking
# et un dossier local dedie pour les artefacts.
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB_PATH}")

if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=MLARTIFACTS_DIR.as_uri(),
    )
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI : {mlflow.get_tracking_uri()}")
print(f"Artifact dir : {MLARTIFACTS_DIR}")
print(f"Experiment   : {EXPERIMENT_NAME}")


2026/06/23 14:03:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/23 14:03:08 INFO mlflow.store.db.utils: Updating database tables


Tracking URI : sqlite:////mnt/c/Users/tanjo/projetcs/fraud-scope/mlflow.db
Artifact dir : /mnt/c/Users/tanjo/projetcs/fraud-scope/mlartifacts
Experiment   : fraud-detection-paytrack


## 2. Chargement et feature engineering

On reconstruit les memes features que dans le notebook de modelisation, y compris les fenetres 1h, 24h et 7 jours.


In [3]:
start = time.time()

train = (
    pl.scan_csv(TRAIN_TRANSACTION_PATH)
    .join(pl.scan_csv(TRAIN_IDENTITY_PATH), on="TransactionID", how="left")
    .sort("TransactionDT")
    .collect()
)

SECONDS_PER_HOUR = 3600
SECONDS_PER_DAY = 24 * SECONDS_PER_HOUR
SECONDS_PER_WEEK = 7 * SECONDS_PER_DAY

train = train.with_columns([
    (pl.col("TransactionDT") // SECONDS_PER_HOUR).cast(pl.Int64).alias("hour_rel"),
    (pl.col("TransactionDT") // SECONDS_PER_DAY).cast(pl.Int64).alias("day_rel"),
    (pl.col("TransactionDT") // SECONDS_PER_WEEK).cast(pl.Int64).alias("week_rel"),
    ((pl.col("TransactionDT") // SECONDS_PER_HOUR) % 24).cast(pl.Int64).alias("hour_of_day_rel"),
])

print(f"Dataset charge : {train.shape}")
print(f"Temps : {time.time() - start:.1f} secondes")


Dataset charge : (590540, 438)
Temps : 104.5 secondes


In [4]:
id_cols = [col for col in ["card1", "card2", "card3", "card5", "addr1"] if col in train.columns]
merchant_cols = [col for col in ["ProductCD", "R_emaildomain"] if col in train.columns]

WINDOW_SPECS = {
    "1h": 60 * 60,
    "24h": 24 * 60 * 60,
    "7d": 7 * 24 * 60 * 60,
}

train = (
    train
    .with_row_count("txn_row_id")
    .with_columns([
        pl.concat_str([pl.col(c).cast(pl.Utf8).fill_null("missing") for c in id_cols], separator="_").alias("customer_proxy"),
        pl.concat_str([pl.col(c).cast(pl.Utf8).fill_null("missing") for c in merchant_cols], separator="_").alias("merchant_proxy"),
    ])
    .sort(["customer_proxy", "TransactionDT", "txn_row_id"])
    .with_columns([
        pl.cum_count("TransactionID").over("customer_proxy").alias("customer_tx_count_cum"),
        pl.col("TransactionAmt").cum_sum().over("customer_proxy").alias("customer_amt_cumsum"),
        pl.cum_count("TransactionID").over(["customer_proxy", "merchant_proxy"]).sub(1).alias("merchant_seen_count_prev"),
    ])
    .with_columns([
        pl.col("customer_tx_count_cum").sub(1).alias("customer_tx_count_prev"),
        (pl.col("customer_amt_cumsum") - pl.col("TransactionAmt")).alias("customer_amt_sum_prev"),
    ])
    .with_columns([
        pl.when(pl.col("customer_tx_count_prev") > 0)
        .then(pl.col("customer_amt_sum_prev") / pl.col("customer_tx_count_prev"))
        .otherwise(None)
        .alias("customer_amt_mean_prev"),
        (pl.col("merchant_seen_count_prev") == 0).cast(pl.Int8).alias("is_new_merchant_for_customer"),
    ])
    .with_columns([
        pl.when(pl.col("customer_tx_count_prev") > 0)
        .then(pl.col("TransactionAmt") / pl.col("customer_amt_mean_prev"))
        .otherwise(None)
        .alias("amt_to_customer_mean_prev"),
    ])
)

history_lookup = train.select([
    "customer_proxy",
    "TransactionDT",
    "customer_tx_count_cum",
    "customer_amt_cumsum",
]).sort("TransactionDT")

helper_cols_to_drop = ["customer_tx_count_cum", "customer_amt_cumsum"]
window_features = []

for label, seconds in WINDOW_SPECS.items():
    before_count_col = f"customer_tx_count_before_{label}"
    before_sum_col = f"customer_amt_sum_before_{label}"
    count_col = f"customer_tx_count_prev_{label}"
    sum_col = f"customer_amt_sum_prev_{label}"

    cutoff_points = (
        train
        .select([
            "txn_row_id",
            "customer_proxy",
            (pl.col("TransactionDT") - seconds).alias("window_start_dt"),
        ])
        .sort("window_start_dt")
    )

    window_start_history = (
        cutoff_points
        .join_asof(
            history_lookup,
            left_on="window_start_dt",
            right_on="TransactionDT",
            by="customer_proxy",
            strategy="backward",
        )
        .select([
            "txn_row_id",
            pl.col("customer_tx_count_cum").fill_null(0).alias(before_count_col),
            pl.col("customer_amt_cumsum").fill_null(0.0).alias(before_sum_col),
        ])
    )

    train = (
        train
        .join(window_start_history, on="txn_row_id", how="left")
        .with_columns([
            pl.max_horizontal([
                pl.col("customer_tx_count_prev") - pl.col(before_count_col),
                pl.lit(0),
            ]).alias(count_col),
            pl.max_horizontal([
                pl.col("customer_amt_sum_prev") - pl.col(before_sum_col),
                pl.lit(0.0),
            ]).alias(sum_col),
        ])
    )

    window_features.extend([count_col, sum_col])
    helper_cols_to_drop.extend([before_count_col, before_sum_col])

engineered_features = [
    "customer_tx_count_prev",
    "customer_amt_mean_prev",
    "amt_to_customer_mean_prev",
    "is_new_merchant_for_customer",
] + window_features

train = train.drop(helper_cols_to_drop).sort("txn_row_id")
print(engineered_features)


['customer_tx_count_prev', 'customer_amt_mean_prev', 'amt_to_customer_mean_prev', 'is_new_merchant_for_customer', 'customer_tx_count_prev_1h', 'customer_amt_sum_prev_1h', 'customer_tx_count_prev_24h', 'customer_amt_sum_prev_24h', 'customer_tx_count_prev_7d', 'customer_amt_sum_prev_7d']


## 3. Split temporel et preprocessing

Le preprocessor est ajuste uniquement sur le train. La validation reste strictement posterieure au train.


In [5]:
compact_numeric_candidates = [
    "TransactionAmt",
    "card1", "card2", "card3", "card5",
    "addr1", "addr2", "dist1", "dist2",
    "C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8", "C9", "C10", "C11", "C12", "C13", "C14",
    "D1", "D2", "D3", "D4", "D5", "D10", "D15",
    "hour_rel", "day_rel", "week_rel", "hour_of_day_rel",
]

numeric_features = [col for col in compact_numeric_candidates if col in train.columns]

candidate_categorical_features = [
    "ProductCD", "card4", "card6", "P_emaildomain", "R_emaildomain",
    "DeviceType", "id_12", "id_15", "id_16", "id_28", "id_29",
    "id_35", "id_36", "id_37", "id_38",
]
categorical_features = [col for col in candidate_categorical_features if col in train.columns]
feature_cols = list(dict.fromkeys(numeric_features + categorical_features + engineered_features))

model_columns = list(dict.fromkeys(feature_cols + ["isFraud", "day_rel", "TransactionID"]))
split_idx = int(train.height * (1 - VALID_SIZE))
train_pd = train.slice(0, split_idx).select(model_columns).to_pandas()
valid_pd = train.slice(split_idx, train.height - split_idx).select(model_columns).to_pandas()

del train
gc.collect()

X_train = train_pd[feature_cols]
y_train = train_pd["isFraud"].astype(int)
X_valid = valid_pd[feature_cols]
y_valid = valid_pd["isFraud"].astype(int)

print(f"Features totales : {len(feature_cols)}")
print(f"Train : {X_train.shape}, jours {train_pd['day_rel'].min()} -> {train_pd['day_rel'].max()}")
print(f"Valid : {X_valid.shape}, jours {valid_pd['day_rel'].min()} -> {valid_pd['day_rel'].max()}")
print(f"Taux fraude train : {y_train.mean():.4%}")
print(f"Taux fraude valid : {y_valid.mean():.4%}")


Features totales : 59
Train : (472432, 59), jours 1 -> 141
Valid : (118108, 59), jours 141 -> 182
Taux fraude train : 3.5135%
Taux fraude valid : 3.4409%


In [6]:
numeric_features_final = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features_final = [col for col in feature_cols if col not in numeric_features_final]

numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features_final),
        ("cat", categorical_pipeline, categorical_features_final),
    ],
    remainder="drop",
)

start = time.time()
X_train_prepared = preprocess.fit_transform(X_train)
X_valid_prepared = preprocess.transform(X_valid)
preprocess_time = time.time() - start
feature_names_prepared = numeric_features_final + categorical_features_final
print(f"Preprocessing : {preprocess_time:.1f} secondes")
print(X_train_prepared.shape, X_valid_prepared.shape)


Preprocessing : 5.8 secondes
(472432, 59) (118108, 59)


## 4. Fonctions de logging MLflow

Chaque run logge :

- parametres modele ;
- metriques principales ;
- temps d'entrainement ;
- temps d'inference ;
- courbe Precision-Recall ;
- matrice de confusion ;
- preprocessor ;
- modele XGBoost.


In [11]:
def evaluate_scores(y_true, scores, threshold=0.25):
    preds = (scores >= threshold).astype(int)
    threshold_name = str(threshold).replace(".", "_")

    return {
        "AUPRC": average_precision_score(y_true, scores),
        f"precision_at_{threshold_name}": precision_score(y_true, preds, zero_division=0),
        f"recall_at_{threshold_name}": recall_score(y_true, preds, zero_division=0),
        f"F1_at_{threshold_name}": f1_score(y_true, preds, zero_division=0),
        "alert_rate": float(preds.mean()),
        "alerts_count": int(preds.sum()),
    }


def save_precision_recall_artifact(y_true, scores, strategy, output_dir):
    precision, recall, _ = precision_recall_curve(y_true, scores)
    auprc = average_precision_score(y_true, scores)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(recall, precision, label=f"{strategy} - AUPRC {auprc:.3f}")
    ax.axhline(np.mean(y_true), color="gray", linestyle="--", label=f"Prevalence {np.mean(y_true):.3f}")
    ax.set_title(f"Precision-Recall - {strategy}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="best")
    path = Path(output_dir) / f"precision_recall_{strategy}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


def save_confusion_matrix_artifact(y_true, scores, strategy, output_dir, threshold=0.25):
    preds = (scores >= threshold).astype(int)
    cm = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(f"Matrice de confusion - {strategy} - seuil {threshold}")
    ax.set_xlabel("Prediction")
    ax.set_ylabel("Vrai label")
    ax.set_xticklabels(["Legitime", "Fraude"])
    ax.set_yticklabels(["Legitime", "Fraude"], rotation=0)
    path = Path(output_dir) / f"confusion_matrix_{strategy}.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return path


def train_strategy(strategy, X_tr, y_tr, X_val, y_val, sampler=None, xgb_params=None):
    params = dict(BASE_XGB_PARAMS)
    if xgb_params:
        params.update(xgb_params)

    start = time.time()
    X_fit, y_fit = X_tr, y_tr
    if sampler is not None:
        X_fit, y_fit = sampler.fit_resample(X_tr, y_tr)

    model = XGBClassifier(**params)
    model.fit(X_fit, y_fit)
    train_time = time.time() - start

    start = time.time()
    scores = model.predict_proba(X_val)[:, 1]
    inference_time = time.time() - start
    inference_ms_per_row = 1000 * inference_time / len(y_val)

    metrics = evaluate_scores(y_val, scores, threshold=DECISION_THRESHOLD)
    metrics.update({
        "train_time_sec": train_time,
        "inference_time_sec": inference_time,
        "inference_ms_per_row": inference_ms_per_row,
        "train_rows_after_sampling": len(y_fit),
        "fraud_rate_after_sampling": float(np.mean(y_fit)),
    })
    return model, scores, metrics


## 5. Entrainement et logging des strategies

Les strategies testees sont les memes que dans le notebook de modelisation.


In [12]:
if MAX_TRAIN_ROWS_FOR_RESAMPLING is not None and len(y_train) > MAX_TRAIN_ROWS_FOR_RESAMPLING:
    start_row = len(y_train) - MAX_TRAIN_ROWS_FOR_RESAMPLING
    X_train_for_sampling = X_train_prepared[start_row:]
    y_train_for_sampling = y_train.iloc[start_row:].to_numpy()
else:
    X_train_for_sampling = X_train_prepared
    y_train_for_sampling = y_train.to_numpy()

y_train_np = y_train.to_numpy()
y_valid_np = y_valid.to_numpy()
scale_pos_weight = (y_train_np == 0).sum() / (y_train_np == 1).sum()
print(f"scale_pos_weight : {scale_pos_weight:.2f}")


scale_pos_weight : 27.46


In [13]:
strategies = [
    {
        "name": "xgboost_baseline",
        "X_tr": X_train_prepared,
        "y_tr": y_train_np,
        "sampler": None,
        "xgb_params": {},
    },
    {
        "name": "xgboost_scale_pos_weight",
        "X_tr": X_train_prepared,
        "y_tr": y_train_np,
        "sampler": None,
        "xgb_params": {"scale_pos_weight": scale_pos_weight},
    },
    {
        "name": "random_undersampling",
        "X_tr": X_train_for_sampling,
        "y_tr": y_train_for_sampling,
        "sampler": RandomUnderSampler(random_state=RANDOM_STATE),
        "xgb_params": {},
    },
    {
        "name": "smote",
        "X_tr": X_train_for_sampling,
        "y_tr": y_train_for_sampling,
        "sampler": SMOTE(random_state=RANDOM_STATE, k_neighbors=5),
        "xgb_params": {},
    },
    {
        "name": "smoteenn",
        "X_tr": X_train_for_sampling,
        "y_tr": y_train_for_sampling,
        "sampler": SMOTEENN(random_state=RANDOM_STATE),
        "xgb_params": {},
    },
]

runs = []
models = {}
scores_by_strategy = {}

for cfg in strategies:
    strategy = cfg["name"]
    with mlflow.start_run(run_name=strategy) as run:
        model, scores, metrics = train_strategy(
            strategy=strategy,
            X_tr=cfg["X_tr"],
            y_tr=cfg["y_tr"],
            X_val=X_valid_prepared,
            y_val=y_valid_np,
            sampler=cfg["sampler"],
            xgb_params=cfg["xgb_params"],
        )

        params_to_log = dict(BASE_XGB_PARAMS)
        params_to_log.update(cfg["xgb_params"])
        params_to_log.update({
            "strategy": strategy,
            "decision_threshold": DECISION_THRESHOLD,
            "feature_set": FEATURE_SET,
            "n_features": len(feature_cols),
            "sampler": type(cfg["sampler"]).__name__ if cfg["sampler"] is not None else "None",
            "max_train_rows_for_resampling": MAX_TRAIN_ROWS_FOR_RESAMPLING,
        })
        mlflow.log_params(params_to_log)
        mlflow.log_metrics(metrics)
        mlflow.set_tags({
            "project": "fraud-scope",
            "dataset": "IEEE-CIS Fraud Detection",
            "split": "temporal",
            "model_family": "xgboost",
        })

        with tempfile.TemporaryDirectory() as tmpdir:
            pr_path = save_precision_recall_artifact(y_valid_np, scores, strategy, tmpdir)
            cm_path = save_confusion_matrix_artifact(y_valid_np, scores, strategy, tmpdir, DECISION_THRESHOLD)
            mlflow.log_artifact(str(pr_path), artifact_path="plots")
            mlflow.log_artifact(str(cm_path), artifact_path="plots")

            preprocessor_path = Path(tmpdir) / "preprocessor.pkl"
            with open(preprocessor_path, "wb") as f:
                pickle.dump(preprocess, f)
            mlflow.log_artifact(str(preprocessor_path), artifact_path="preprocessing")

            features_path = Path(tmpdir) / "feature_columns.json"
            features_path.write_text(json.dumps(feature_cols, indent=2))
            mlflow.log_artifact(str(features_path), artifact_path="metadata")

        mlflow.xgboost.log_model(model, artifact_path="xgboost_model")

        row = {"strategy": strategy, "run_id": run.info.run_id, **metrics}
        runs.append(row)
        models[strategy] = model
        scores_by_strategy[strategy] = scores
        print(strategy, row)

results_df = pd.DataFrame(runs).sort_values("AUPRC", ascending=False).reset_index(drop=True)
display(results_df)

results_path = PROJECT_ROOT / "docs" / "mlflow_runs_summary.csv"
results_df.to_csv(results_path, index=False)
print(f"Resume des runs sauvegarde : {results_path}")


2026/06/23 14:11:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost_baseline {'strategy': 'xgboost_baseline', 'run_id': '6e7f31edf29b4a5789dadcd9b5212cd1', 'AUPRC': 0.47533407651431325, 'precision_at_0_25': 0.5842618384401114, 'recall_at_0_25': 0.41289370078740156, 'F1_at_0_25': 0.48385236447520186, 'alert_rate': 0.024316727063365733, 'alerts_count': 2872, 'train_time_sec': 9.35679316520691, 'inference_time_sec': 0.05176520347595215, 'inference_ms_per_row': 0.0004382870209973257, 'train_rows_after_sampling': 472432, 'fraud_rate_after_sampling': 0.03513521522674162}


2026/06/23 14:11:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


xgboost_scale_pos_weight {'strategy': 'xgboost_scale_pos_weight', 'run_id': '2fa7a9457ea748b9b774d975383e0b79', 'AUPRC': 0.45250590589959616, 'precision_at_0_25': 0.07232994767620807, 'recall_at_0_25': 0.9251968503937008, 'F1_at_0_25': 0.13417071081929774, 'alert_rate': 0.4401395333084973, 'alerts_count': 51984, 'train_time_sec': 6.490321159362793, 'inference_time_sec': 0.04048037528991699, 'inference_ms_per_row': 0.0003427403333382751, 'train_rows_after_sampling': 472432, 'fraud_rate_after_sampling': 0.03513521522674162}


2026/06/23 14:12:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

smote {'strategy': 'smote', 'run_id': 'd7a4506a62324db9857519deff270ddd', 'AUPRC': 0.41369148215296103, 'precision_at_0_25': 0.19588653566786451, 'recall_at_0_25': 0.6304133858267716, 'F1_at_0_25': 0.2988975091874234, 'alert_rate': 0.11073762996579402, 'alerts_count': 13079, 'train_time_sec': 7.076756000518799, 'inference_time_sec': 0.053992509841918945, 'inference_ms_per_row': 0.00045714523861143145, 'train_rows_after_sampling': 288722, 'fraud_rate_after_sampling': 0.5}


2026/06/23 14:14:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


smoteenn {'strategy': 'smoteenn', 'run_id': 'c71f1a8334e445c69710b0be38a71c51', 'AUPRC': 0.42650658985075107, 'precision_at_0_25': 0.18682241029586627, 'recall_at_0_25': 0.6572342519685039, 'F1_at_0_25': 0.290942759108981, 'alert_rate': 0.12105022521759745, 'alerts_count': 14297, 'train_time_sec': 132.85127019882202, 'inference_time_sec': 0.06004476547241211, 'inference_ms_per_row': 0.0005083886398246698, 'train_rows_after_sampling': 268243, 'fraud_rate_after_sampling': 0.5325134299869894}


,strategy,run_id,AUPRC,precision_at_0_25,recall_at_0_25,F1_at_0_25,alert_rate,alerts_count,train_time_sec,inference_time_sec,inference_ms_per_row,train_rows_after_sampling,fraud_rate_after_sampling
0,xgboost_baseline,6e7f31edf29b4a5789dadcd9b5212cd1,0.4753,0.5843,0.4129,0.4839,0.0243,2872,9.3568,0.0518,0.0004,472432,0.0351
1,xgboost_scale_pos_weight,2fa7a9457ea748b9b774d975383e0b79,0.4525,0.0723,0.9252,0.1342,0.4401,51984,6.4903,0.0405,0.0003,472432,0.0351
2,random_undersampling,a68a1286189d47cfb1d72feb24b3f132,0.4523,0.0787,0.8999,0.1448,0.3933,46446,1.7614,0.0556,0.0005,11278,0.5000
3,smoteenn,c71f1a8334e445c69710b0be38a71c51,0.4265,0.1868,0.6572,0.2909,0.1211,14297,132.8513,0.0600,0.0005,268243,0.5325
4,smote,d7a4506a62324db9857519deff270ddd,0.4137,0.1959,0.6304,0.2989,0.1107,13079,7.0768,0.0540,0.0005,288722,0.5000


Resume des runs sauvegarde : /mnt/c/Users/tanjo/projetcs/fraud-scope/docs/mlflow_runs_summary.csv


## 6. Selection du meilleur modele et validation gate

Le meilleur modele est choisi par AUPRC. La gate valide ensuite que le modele respecte les seuils minimums de qualite et de latence.


In [15]:
best = results_df.iloc[0].to_dict()
best_strategy = best["strategy"]
best_run_id = best["run_id"]

threshold_name = str(DECISION_THRESHOLD).replace(".", "_")
recall_metric_name = f"recall_at_{threshold_name}"

passed_gate = (
    best["AUPRC"] >= GATE_MIN_AUPRC
    and best[recall_metric_name] >= GATE_MIN_RECALL
    and best["alert_rate"] <= GATE_MAX_ALERT_RATE
    and best["inference_ms_per_row"] <= GATE_MAX_INFERENCE_MS_PER_ROW
)

gate_report = {
    "best_strategy": best_strategy,
    "best_run_id": best_run_id,
    "gate_status": "PASSED" if passed_gate else "FAILED",
    "AUPRC": best["AUPRC"],
    "AUPRC_threshold": GATE_MIN_AUPRC,
    recall_metric_name: best[recall_metric_name],
    "recall_threshold": GATE_MIN_RECALL,
    "alert_rate": best["alert_rate"],
    "max_alert_rate": GATE_MAX_ALERT_RATE,
    "inference_ms_per_row": best["inference_ms_per_row"],
    "max_inference_ms_per_row": GATE_MAX_INFERENCE_MS_PER_ROW,
}

display(pd.Series(gate_report).to_frame("validation_gate"))

with mlflow.start_run(run_id=best_run_id):
    mlflow.set_tag("validation_gate_status", gate_report["gate_status"])
    mlflow.log_metrics({
        "gate_min_AUPRC": GATE_MIN_AUPRC,
        "gate_min_recall": GATE_MIN_RECALL,
        "gate_max_alert_rate": GATE_MAX_ALERT_RATE,
        "gate_max_inference_ms_per_row": GATE_MAX_INFERENCE_MS_PER_ROW,
    })
    with tempfile.TemporaryDirectory() as tmpdir:
        gate_path = Path(tmpdir) / "validation_gate_report.json"
        gate_path.write_text(json.dumps(gate_report, indent=2))
        mlflow.log_artifact(str(gate_path), artifact_path="validation")

print(json.dumps(gate_report, indent=2))

,validation_gate
best_strategy,xgboost_baseline
best_run_id,6e7f31edf29b4a5789dadcd9b5212cd1
gate_status,PASSED
AUPRC,0.4753
AUPRC_threshold,0.4500
recall_at_0_25,0.4129
recall_threshold,0.4000
alert_rate,0.0243
max_alert_rate,0.0500
inference_ms_per_row,0.0004


{
  "best_strategy": "xgboost_baseline",
  "best_run_id": "6e7f31edf29b4a5789dadcd9b5212cd1",
  "gate_status": "PASSED",
  "AUPRC": 0.47533407651431325,
  "AUPRC_threshold": 0.45,
  "recall_at_0_25": 0.41289370078740156,
  "recall_threshold": 0.4,
  "alert_rate": 0.024316727063365733,
  "max_alert_rate": 0.05,
  "inference_ms_per_row": 0.0004382870209973257,
  "max_inference_ms_per_row": 50.0
}


## 7. Enregistrement du meilleur modele

Si la gate passe, le modele est enregistre dans le registry local MLflow.

Pour la baseline et `scale_pos_weight`, on peut enregistrer une pipeline sklearn complete `preprocess + model`. Pour les strategies avec resampling, on enregistre le modele XGBoost du run et le preprocessor reste disponible comme artefact.


In [17]:
registered_version = None

if passed_gate:
    best_model = models[best_strategy]

    with mlflow.start_run(run_id=best_run_id):
        model_info = mlflow.xgboost.log_model(
            xgb_model=best_model,
            artifact_path="production_candidate_xgboost_model",
            registered_model_name=REGISTERED_MODEL_NAME,
        )
        print("Modele logge :", model_info.model_uri)

    client = MlflowClient()
    versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
    latest_versions = sorted(versions, key=lambda v: int(v.version), reverse=True)

    if latest_versions:
        registered_version = latest_versions[0]

        client.set_model_version_tag(
            name=REGISTERED_MODEL_NAME,
            version=registered_version.version,
            key="validation_gate_status",
            value="PASSED",
        )
        client.set_model_version_tag(
            name=REGISTERED_MODEL_NAME,
            version=registered_version.version,
            key="source_run_id",
            value=best_run_id,
        )
        client.set_model_version_tag(
            name=REGISTERED_MODEL_NAME,
            version=registered_version.version,
            key="candidate_stage",
            value="Staging",
        )

        print(f"Modele enregistre : {REGISTERED_MODEL_NAME} v{registered_version.version}")
else:
    print("Gate echouee : aucun modele enregistre dans le registry.")

2026/06/23 14:20:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'paytrack-fraud-xgboost'.
Created version '1' of model 'paytrack-fraud-xgboost'.


Modele logge : models:/m-6cdd7949e8504edc91fdf868bbd5ba42
Modele enregistre : paytrack-fraud-xgboost v1


## 8. Conclusion MLOps

A reporter dans le README / rapport :

- tous les runs sont traces dans MLflow ;
- les metriques principales sont journalisees ;
- les courbes PR et matrices de confusion sont disponibles comme artefacts ;
- le meilleur modele est selectionne par AUPRC ;
- une validation gate evite de promouvoir un modele qui degrade la performance ou la latence ;
- le modele valide est enregistre dans le Model Registry local.

Commande pour ouvrir l'UI :

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 8080
```
